In [4]:
import shap

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.base import clone

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
# HELPER FUNCTIONS

def make_names_dict():
    # FIXME: move this to some kind of config
    # Dictionary for f-string construction in model paths and in text on ROC plots
    model_names = ['RandomForestClassifier', 'SVC', 'LogisticRegression', 'XGBClassifier']
    model_names_dict = {name:{'full':'', 'short':''} for name in model_names}

    model_names_dict['RandomForestClassifier']['full'] = 'Random Forest'
    model_names_dict['RandomForestClassifier']['short'] = 'RF'

    model_names_dict['SVC']['full'] = 'Support Vector Machine'
    model_names_dict['SVC']['short'] = 'SVM'

    model_names_dict['LogisticRegression']['full'] = 'Logistic Regression'
    model_names_dict['LogisticRegression']['short'] = 'LR'

    model_names_dict['XGBClassifier']['full'] = 'XGBoost'
    model_names_dict['XGBClassifier']['short'] = 'XGB'

    return model_names_dict

def compose_windows(tissue, windows = ["06-08", "10-12", "14-16"]
):
    """
    Concatenate window-specific DataFrames and generate a `composite` vector for stratification
    """
    Xs, ys = [], []
    for idx, w in enumerate(windows):
        curr_X = pd.read_csv(f"data/training/hrs{w}/data_diff_hrs{w}.csv", index_col=0)
        curr_y = pd.read_csv(f"data/training/hrs{w}/y_{tissue}.csv", index_col=0).iloc[:, 0]
        curr_X['window'] = idx

        Xs.append(curr_X)
        ys.append(curr_y)

    y_new = pd.concat(ys, axis=0)
    X_new = pd.concat(Xs, axis=0)

    composite = pd.Categorical(list(zip(X_new['window'], y_new))).codes
    num_values = len(pd.Series(composite).unique())
    #print(f"Created a composite vector with {num_values} distinct values")

    X_new.drop('window', axis=1, inplace=True) # we don't want to use 'window' for prediction

    return X_new, y_new, composite

In [ ]:
"""
ścieżka anotacji motywów
/home/ajank/enhancer-promoter-interactions/data/calderon_data/motif_names.[rds|tsv]
"""

def shap_analysis_with_beeswarm(
    classifier_path: str,
    tissue: str,
    motif_annotations_path: str = None,
    windows: list[str] = ["06-08", "10-12", "14-16"],
    top_n_motifs: int = 10,
    num_shap_samples: int = None,
    random_state:int = 0
):
    """
    Perform SHAP-based interpretability analysis for a pretrained
    time-agnostic chromatin loop classifier.

    This function:
    1. Loads a pretrained classifier from disk (.pkl)
    2. Reconstructs the time-agnostic dataset for a given tissue
    3. (optional) Draws a stratified SHAP background/sample using (window, label) composite
    4. Computes SHAP values using an explainer appropriate to the model class
    5. Produces and saves a SHAP beeswarm plot
    6. Constructs and saves a dataframe containing SHAP statistics for ALL motifs:
       - mean SHAP value
       - absolute mean SHAP value
       - standard deviation of SHAP values

    Args:
        classifier_path (str): Path to a trained time-agnostic classifier (.pkl file)

        tissue (str): Tissue name (must match label file naming convention)

        windows (list[str]): Time windows used to construct the time-agnostic dataset

        top_n_motifs (int): Number of motifs to show in the bar plot

        num_shap_samples (int): (Optional) number of stratified samples used for SHAP computation

        random_state (int): Random seed for reproducibility

    Returns:    
        shap_df (pandas.DataFrame): DataFrame with SHAP statistics for all motifs
    """

    # Load model
    with open(classifier_path, "rb") as f:
        model = pickle.load(f)

    model_names = make_names_dict()
    model_key = type(model).__name__
    model_name = model_names[model_key]['full']
    model_short = model_names[model_key]['short']

    # Load data
    X, y, composite = compose_windows(tissue, windows)

    # (Optional) draw a stratified SHAP sample to perform the analysis on a subset of the data
    if num_shap_samples:
        X_shap = stratified_shap_sample(
            X, composite,
            n_samples=num_shap_samples,
            random_state=random_state
        )
    else:
        X_shap = X


    def predict_proba_pos(model, X):
        """Return P(y=1) only."""
        return model.predict_proba(X)[:, 1]
        
    if model_key in ["RandomForestClassifier", "XGBClassifier"]:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_shap)

        # Normalize TreeExplainer output
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        elif shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

    else:
        # IMPORTANT: explain only P(y=1)
        explainer = shap.KernelExplainer(
            lambda X: predict_proba_pos(model, X),
            X_shap
        )
        shap_values = explainer.shap_values(X_shap, nsamples=num_shap_samples)

    # Beeswarm plot
    plt.figure(figsize=(8, 8))
    shap.summary_plot(shap_values, X_shap, show=False)

    if num_shap_samples:
        plt.title(f"{model_name} ({tissue}) beeswarm SHAP feature importance ({num_shap_samples} stratified samples)")
    else:
        plt.title(f"{model_name} for {tissue}: beeswarm SHAP feature importance (top {top_n_motifs} most important motifs)")
     
    
    plt.tight_layout()

    figures_path = f"results/figures/shap/{model_short}"
    os.makedirs(figures_path, exist_ok=True)
    beeswarm_path = os.path.join(
        figures_path, f"beeswarm_{model_short}_{tissue}.png"
    )
    plt.savefig(beeswarm_path, dpi=300)
    #plt.show()

    # Aggregate statistics
    mean_vals = shap_values.mean(axis=0)
    std_vals = shap_values.std(axis=0)
    abs_mean_vals = np.abs(mean_vals)
    mean_abs_vals = np.abs(shap_values).mean(axis=0)

    shap_df = pd.DataFrame({
        "motif": X.columns,
        "mean_importance": mean_vals,
        "abs_mean_importance": abs_mean_vals,
        "mean_abs_importance": mean_abs_vals,
        "std_importance": std_vals
    }).sort_values("abs_mean_importance", ascending=False)

    data_path = f"results/data/shap/{model_short}"
    os.makedirs(data_path, exist_ok=True)
    df_path = os.path.join(
        data_path, f"shap_table_{model_short}_{tissue}.csv"
    )
    shap_df.to_csv(df_path, index=False)

    # Bar plot (top N)
    top = shap_df.head(top_n_motifs)

    x = np.arange(len(top))
    means = top["mean_importance"].values
    stds = top["std_importance"].values
    labels = top["motif"].values

    colors = ["green" if m > 0 else "red" for m in means]

    fig, ax = plt.subplots(figsize=(12, 6))

    # Bars
    ax.bar(
        x,
        np.abs(means),
        color=colors,
        alpha=0.8,
        width=0.8
    )

    # Error bars (same style as ROC plot)
    ax.errorbar(
        x,
        np.abs(means),
        yerr=stds,
        fmt='.',
        capsize=6,
        color='black',
        alpha=0.8
    )

    # Axis formatting
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel("Absolute Mean SHAP value")
    ax.set_xlabel("Motif")
    ax.set_title(f"{model_name} ({tissue}) absolute mean SHAP feature importance ({num_shap_samples} stratified samples)")

    plt.tight_layout()

    barplot_path = os.path.join(
        figures_path, f"barplot_{model_short}_{tissue}.png"
    )
    plt.savefig(barplot_path, dpi=300)
    #plt.show()

    return shap_df


In [ ]:
tissue = "Neuroblasts"
windows=["06-08", "10-12", "14-16"]
X, y, composite = compose_windows(tissue, windows)

sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=100,
    random_state=1
)
#next(sss.split(X, composite))

In [5]:
tissues = ["Neuroblasts", "Neurons", "Glia"]
short_names = ['RF', 'SVM', 'LR', 'XGB']


for model in ['SVM', 'LR']:
    for t in tissues: 
        print(f"SHAP analysis for model {model} for tissue {t}...")
        shap_df = shap_analysis_with_barplot_and_beeswarm(
            classifier_path=f"results/models/time_agnostic/all_data/{model}_{t}.pkl",
            tissue=t,
            top_n_motifs=20,
            num_shap_samples=100,
        )


for model in ['RF', 'XGB']:
    for t in tissues: 
        print(f"SHAP analysis for model {model} for tissue {t}...")
        shap_df = shap_analysis_with_barplot_and_beeswarm(
            classifier_path=f"results/models/time_agnostic/all_data/{model}_{t}.pkl",
            tissue=t,
            top_n_motifs=20,
            num_shap_samples=300,
        )
